# ENG 6010 Test 2 - Q3: RL evidence search (PPO) on Colab

Trains the PPO search policy for the Go2 in the house-185 crime scene (`CrimeSceneSearchEnv`), saving
**everything to Google Drive** so a disconnected runtime can **resume** from the last checkpoint.

* The environment is pure numpy (the MuJoCo scenes were exported to `q3_rl/layouts/`), so no MuJoCo or GPU is
  needed for training. A **CPU runtime is fine** (the MLP policy is tiny; the GPU does not help PPO here).
* Hyper-parameters: `config/q3_ppo.yaml`; environment and reward weights: `config/q3_env.yaml`.
* Re-run the *Train* cell after a disconnect: `--resume` continues from `checkpoints/ppo_<steps>_steps.zip`
  with the step counter, Monitor logs and EvalCallback history intact.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/eng6010_test2_q3'   # checkpoints, best_model.zip, TensorBoard, logs
import os; os.makedirs(OUT, exist_ok=True); print(OUT)

In [ ]:
BRANCH = 'main'
!test -d /content/repo || git clone -q --depth 1 -b $BRANCH https://github.com/CDJ7/Unitree-Dog-Claude-Dog-.git /content/repo
%cd /content/repo/Test2
!pip -q install "stable-baselines3>=2.3" "gymnasium>=1.0" scikit-image pyyaml tensorboard pandas matplotlib

## 1. Sanity checks: the environment passes the Gymnasium / SB3 checkers; baselines on the test layouts

In [ ]:
!python -m pytest -q tests/test_q3_env.py
!python q3_rl/evaluate.py --policies random,greedy --episodes 60 --ppo-dir $OUT

## 2. TensorBoard (live while training)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir $OUT/tb

## 3. Train (re-run this cell to resume after a disconnect)
`--timesteps` is the **total** target; with `--resume` only the remaining steps are trained.
The reported run (v4, with the training curriculum) used 6M steps, about 53 min of training on 4 CPU cores here;
a 2-core Colab CPU runtime is slower. For a quick check, use e.g. `--timesteps 24576`.

In [ ]:
!python q3_rl/train_ppo.py --out $OUT --resume --timesteps 6000000 --n-envs 8

## 4. Evaluate PPO vs random, greedy and planner baselines (60 identical test episodes each) + learning curves + trajectories

In [ ]:
!python q3_rl/evaluate.py --ppo-dir $OUT --episodes 60
from IPython.display import Image, Markdown, display
display(Markdown(open('results/q3/comparison.md').read()))
for f in ['learning_curves.png', 'trajectories.png']:
    display(Image(f'results/q3/{f}'))

## 5. (Optional) Deploy on the full MuJoCo Go2
Needs the MuJoCo simulator submodule and an offscreen GL backend; this works on Colab but takes a few minutes to set up.
The policy's (vx, vy, yaw_rate) commands drive the simulator's learned locomotion controller (Q2).

In [ ]:
!git submodule update --init --depth 1 ../Phase_2_Investigator/third_party/robot-unitree-go2-mujoco
!pip -q install mujoco==3.3.7 onnxruntime pillow
!apt-get -qq install -y libegl1 libegl-mesa0 libosmesa6 libgl1-mesa-dri > /dev/null
%env MUJOCO_GL=egl
!python q3_rl/deploy_mujoco.py --layout s2 --model $OUT/best_model.zip
display(Image('results/q3/deploy_s2.gif'))

## 6. Copy results to Drive
The figures and metrics are in `results/q3/`; copy them next to the checkpoints.

In [ ]:
!mkdir -p $OUT/results && cp -r results/q3/*.png results/q3/*.json results/q3/*.md results/q3/*.csv $OUT/results/ 2>/dev/null; ls $OUT/results